In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import mygene

# ---------------------------------------------------------
# PARAMETERS (EASILY CONFIGURABLE)
# ---------------------------------------------------------

EXPR_FILE = "S1.csv"
GENE_LIST_FILE = "network_genes.csv"
PATIENT_ID = "GSM158xxxx"

NORMAL_SAMPLES = [
    "GSM1589130", "GSM1589132", "GSM1589135", "GSM1589136",
    "GSM1589139", "GSM1589142", "GSM1589144", "GSM1589145",
    "GSM1589148", "GSM1589150", "GSM1589151"
]

P_VALUE = 0.95

OUTPUT_FILE = f"TNBC_{PATIENT_ID}_BOOLEAN_p95.csv"

# ---------------------------------------------------------
# 1) DATA LOADING
# ---------------------------------------------------------

expr = pd.read_csv(EXPR_FILE, index_col=0)

# ---------------------------------------------------------
# 2) PROBE-TO-GENE SYMBOL ANNOTATION
# ---------------------------------------------------------

mg = mygene.MyGeneInfo()

probe_ids = expr.index.tolist()
annotation = mg.querymany(
    probe_ids,
    scopes="reporter",
    fields="symbol",
    species="human"
)

probe_to_gene = {
    item["query"]: item.get("symbol", None)
    for item in annotation
}

expr["GeneSymbol"] = expr.index.map(probe_to_gene)
expr = expr.dropna(subset=["GeneSymbol"])

# ---------------------------------------------------------
# 3) GENE-LEVEL AGGREGATION (MEAN OF PROBES)
# ---------------------------------------------------------

expr_by_gene = expr.groupby("GeneSymbol").mean()
expr_by_gene.index = expr_by_gene.index.str.strip().str.upper()

# Rename columns to match GSMxxxx nomenclature
expr_by_gene.columns = expr_by_gene.columns.str.extract(r"(GSM\d+)", expand=False)

# ---------------------------------------------------------
# 4) COMPUTATION OF DIFFERENTIAL EXPRESSION (Tumor − Mean of Normals)
# ---------------------------------------------------------

tumor_expr = expr_by_gene[PATIENT_ID]
normal_mean = expr_by_gene[NORMAL_SAMPLES].mean(axis=1)

diff = tumor_expr - normal_mean

# ---------------------------------------------------------
# 5) ESTIMATION OF DIFFERENTIAL DISTRIBUTION AND CVC (Pires Method)
# ---------------------------------------------------------

mu, sigma = norm.fit(diff)
critical_value = norm.ppf(P_VALUE, loc=mu, scale=sigma)

# ---------------------------------------------------------
# 6) BOOLEANIZATION
# ---------------------------------------------------------

boolean = (diff > critical_value).astype(int)

df_boolean = pd.DataFrame({
    "GeneSymbol": diff.index,
    "Diff": diff.values,
    "Boolean": boolean.values
})

# ---------------------------------------------------------
# 7) NETWORK GENE EXTRACTION
# ---------------------------------------------------------

gene_list = pd.read_csv(GENE_LIST_FILE)
gene_list["GeneSymbol"] = gene_list["GeneSymbol"].str.strip().str.upper()

df_network = df_boolean[df_boolean["GeneSymbol"].isin(gene_list["GeneSymbol"])]

# ---------------------------------------------------------
# 8) RESULT SERIALIZATION
# ---------------------------------------------------------

df_network.to_csv(OUTPUT_FILE, index=False)

# ---------------------------------------------------------
# 9) SUMMARY REPORT
# ---------------------------------------------------------

print("=== BOOLEANIZATION COMPLETED ===")
print("TNBC Patient:", PATIENT_ID)
print("Total number of genes:", len(df_boolean))
print("Number of network genes:", len(df_network))
print("Active genes (1):", df_network["Boolean"].sum())
print("CVC Threshold:", critical_value)
print("Output file saved:", OUTPUT_FILE)